# Mortgage Data Ingestion


Load, inspect, clean, and prepare the 2019 Freddie Mac Single-Family Loan-Level Dataset sample for analysis and modeling.


## 1. Imports


In [94]:
import pandas as pd


## 2. Origination Data


### 2.1 Load Origination Data


Load the 2019 Freddie Mac origination sample, where each row represents one mortgage at origination.


In [95]:
orig = pd.read_csv(
    "../data/raw_mortgage/sample_2019/sample_orig_2019.txt",
    sep="|",
    header=None,
    low_memory=False
)


### 2.2 Assign Column Names


In [96]:
orig_columns = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer",
    "maturity_date",
    "msa",
    "mi_percent",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "original_interest_rate",
    "channel",
    "prepayment_penalty",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_id",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "super_conforming_flag",
    "pre_harp_loan_id",
    "special_eligibility_program",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "vantage_score"
]

orig.columns = orig_columns


### 2.3 Initial Inspection


In [97]:
orig.head()


,credit_score,first_payment_date,first_time_homebuyer,maturity_date,msa,mi_percent,num_units,occupancy_status,cltv,dti,...,original_loan_term,num_borrowers,seller_name,super_conforming_flag,pre_harp_loan_id,special_eligibility_program,harp_indicator,property_valuation_method,interest_only_indicator,vantage_score
0,741,201903,N,204902,NaN,0,1,P,80,33,...,360,2,OTHER,N,NaN,NaN,N,7,N,9999
1,731,201903,N,204902,13460.0,0,1,P,77,44,...,360,1,OTHER,N,NaN,NaN,N,7,N,9999
2,722,201903,N,204902,NaN,30,1,P,95,41,...,360,2,OTHER,N,NaN,NaN,N,7,N,9999
3,729,201903,N,204902,NaN,25,1,P,87,17,...,360,1,OTHER,N,NaN,NaN,N,7,N,9999
4,773,201903,N,204902,33700.0,0,1,P,29,43,...,360,2,OTHER,N,NaN,NaN,N,7,N,9999


In [98]:
orig.shape


(50000, 31)

In [99]:
orig.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   credit_score                 50000 non-null  int64  
 1   first_payment_date           50000 non-null  int64  
 2   first_time_homebuyer         50000 non-null  object 
 3   maturity_date                50000 non-null  int64  
 4   msa                          45426 non-null  float64
 5   mi_percent                   50000 non-null  int64  
 6   num_units                    50000 non-null  int64  
 7   occupancy_status             50000 non-null  object 
 8   cltv                         50000 non-null  int64  
 9   dti                          50000 non-null  int64  
 10  original_upb                 50000 non-null  int64  
 11  ltv                          50000 non-null  int64  
 12  original_interest_rate       50000 non-null  float64
 13  channel         

### 2.4 Missing Value Handling


Freddie Mac uses sentinel values to represent missing or unavailable information. Convert these codes to standard missing values so they are not interpreted as real measurements.


In [100]:
missing_codes = {
    "credit_score": [9999, "9999"],
    "mi_percent": [999, "999"],
    "num_units": [99, "99"],
    "occupancy_status": [9, "9"],
    "cltv": [999, "999"],
    "dti": [999, "999"],
    "ltv": [999, "999"],
    "channel": [9, "9"],
    "property_type": [99, "99"],
    "postal_code": [0, "000"],
    "loan_purpose": [9, "9"],
    "num_borrowers": [99, "99"],
    "property_valuation_method": [7, "7"],
    "vantage_score": [9999, "9999"]
}

for col, sentinels in missing_codes.items():
    orig[col] = orig[col].replace(sentinels, pd.NA)


In [101]:
orig[list(missing_codes.keys())].isna().sum()


credit_score                    18
mi_percent                       0
num_units                        0
occupancy_status                 0
cltv                             1
dti                             31
ltv                              0
channel                          0
property_type                    0
postal_code                      0
loan_purpose                     0
num_borrowers                    0
property_valuation_method    44564
vantage_score                50000
dtype: int64

**Observation:** Missing-value codes were converted to standard missing values. VantageScore is unavailable for the full 2019 sample, while property valuation method is missing for most observations; these fields should not be treated as fully observed predictors.


## 3. Monthly Performance Data


### 3.1 Load Performance Data


Load the 2019 Freddie Mac monthly performance file. Each mortgage can appear in multiple rows as its status is reported over time.


In [102]:
perf = pd.read_csv(
    "../data/raw_mortgage/sample_2019/sample_perf_2019.txt",
    sep="|",
    header=None,
    low_memory=False
)


### 3.2 Assign Column Names


In [103]:
perf_columns = [
    "loan_id",
    "monthly_reporting_period",
    "current_actual_upb",
    "delinquency_status",
    "loan_age",
    "remaining_months_to_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_deferred_upb",
    "due_date_last_paid_installment",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_preservation_costs",
    "taxes_insurance",
    "misc_expenses",
    "actual_loss",
    "modification_cost",
    "step_modification_flag",
    "deferred_payment_plan",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status",
    "current_month_modification_cost",
    "interest_bearing_upb",
    "alternative_delinquency_resolution",
    "alternative_delinquency_resolution_count",
    "total_deferral_amount"
]

perf.columns = perf_columns


### 3.3 Initial Inspection


In [104]:
perf.shape


(1934614, 35)

In [105]:
perf.head()


,loan_id,monthly_reporting_period,current_actual_upb,delinquency_status,loan_age,remaining_months_to_maturity,defect_settlement_date,modification_flag,zero_balance_code,zero_balance_effective_date,...,estimated_ltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_status,current_month_modification_cost,interest_bearing_upb,alternative_delinquency_resolution,alternative_delinquency_resolution_count,total_deferral_amount
0,F19Q10000056,201902,372000.0,00,0,360,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,372000.0,7,OTHER,NaN
1,F19Q10000056,201903,372000.0,00,1,359,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,372000.0,7,OTHER,NaN
2,F19Q10000056,201904,371000.0,00,2,358,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,371000.0,7,OTHER,NaN
3,F19Q10000056,201905,370000.0,00,3,357,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,370000.0,7,OTHER,NaN
4,F19Q10000056,201906,370000.0,00,4,356,NaN,NaN,NaN,NaN,...,76,NaN,NaN,NaN,NaN,NaN,370000.0,7,OTHER,NaN


In [106]:
perf.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1934614 entries, 0 to 1934613
Data columns (total 35 columns):
 #   Column                                    Dtype  
---  ------                                    -----  
 0   loan_id                                   object 
 1   monthly_reporting_period                  int64  
 2   current_actual_upb                        float64
 3   delinquency_status                        object 
 4   loan_age                                  int64  
 5   remaining_months_to_maturity              int64  
 6   defect_settlement_date                    float64
 7   modification_flag                         object 
 8   zero_balance_code                         float64
 9   zero_balance_effective_date               float64
 10  current_interest_rate                     float64
 11  current_deferred_upb                      float64
 12  due_date_last_paid_installment            float64
 13  mi_recoveries                             float64
 14  ne

### 3.4 Delinquency Status


In [107]:
perf["delinquency_status"].value_counts(
    dropna=False
).sort_index()


delinquency_status
00    1883898
01      18410
02       6678
03       4341
04       3153
       ...   
60          1
61          1
62          1
63          1
RA        137
Name: count, Length: 65, dtype: int64

Freddie Mac records delinquency status using numeric codes for months delinquent along with non-numeric statuses. Convert numeric statuses for severity calculations while retaining the original field for investigation of non-numeric outcomes.


In [108]:
perf["delinquency_numeric"] = pd.to_numeric(
    perf["delinquency_status"],
    errors="coerce"
)


In [109]:
perf[
    ["delinquency_status", "delinquency_numeric"]
].value_counts(dropna=False).sort_index()


delinquency_status  delinquency_numeric
00                  0.0                    1883898
01                  1.0                      18410
02                  2.0                       6678
03                  3.0                       4341
04                  4.0                       3153
                                            ...   
60                  60.0                         1
61                  61.0                         1
62                  62.0                         1
63                  63.0                         1
RA                  NaN                        137
Name: count, Length: 65, dtype: int64

## 4. Loan-Level Delinquency Exploration


### 4.1 Maximum Delinquency by Loan


As an initial target-definition exercise, summarize each loan's maximum observed delinquency across its full available performance history. This is used only to compare delinquency thresholds; the final prediction target uses a fixed 24-month performance window.


In [110]:
loan_delinquency = (
    perf.groupby("loan_id")["delinquency_numeric"]
    .max()
    .reset_index()
    .rename(columns={"delinquency_numeric": "max_delinquency"})
)

loan_delinquency.head()


,loan_id,max_delinquency
0,F19Q10000056,0.0
1,F19Q10000060,0.0
2,F19Q10000084,0.0
3,F19Q10000106,0.0
4,F19Q10000189,0.0


In [111]:
loan_delinquency.shape


(50000, 2)

### 4.2 Delinquency Thresholds


In [112]:
loan_delinquency["ever_30_plus"] = loan_delinquency["max_delinquency"] >= 1
loan_delinquency["ever_60_plus"] = loan_delinquency["max_delinquency"] >= 2
loan_delinquency["ever_90_plus"] = loan_delinquency["max_delinquency"] >= 3


In [113]:
loan_delinquency[
    ["ever_30_plus", "ever_60_plus", "ever_90_plus"]
].mean()


ever_30_plus    0.13276
ever_60_plus    0.06988
ever_90_plus    0.05572
dtype: float64

In [114]:
reached_30 = loan_delinquency["ever_30_plus"].sum()
reached_60 = loan_delinquency["ever_60_plus"].sum()
reached_90 = loan_delinquency["ever_90_plus"].sum()

print(f"30+ delinquency: {reached_30:,}")
print(f"60+ delinquency: {reached_60:,}")
print(f"90+ delinquency: {reached_90:,}")
print(f"30+ progressing to 60+: {reached_60 / reached_30:.1%}")
print(f"60+ progressing to 90+: {reached_90 / reached_60:.1%}")


30+ delinquency: 6,638
60+ delinquency: 3,494
90+ delinquency: 2,786
30+ progressing to 60+: 52.6%
60+ progressing to 90+: 79.7%


**Observation:** Across the full observed performance history, 13.28% of loans reached 30+ days delinquent, 6.99% reached 60+, and 5.57% reached 90+. About 52.6% of loans reaching 30+ progressed to 60+, while 79.7% of those reaching 60+ progressed to 90+. Based on this progression pattern, 90+ delinquency was selected as the serious-delinquency threshold for the prediction target.


### 4.3 Zero-Balance Outcomes


Zero-balance codes identify why a mortgage leaves the active performance dataset, such as prepayment, sale, or other termination events. Inspect these outcomes before defining the fixed prediction window.


In [115]:
perf["zero_balance_code"].value_counts(
    dropna=False
).sort_index()


zero_balance_code
1.0       35551
2.0          18
3.0           4
9.0          14
15.0         13
16.0         31
96.0        123
NaN     1898860
Name: count, dtype: int64

## 5. Calendar-Based Target Construction


### 5.1 Construct Calendar Loan Age


Freddie Mac's reported `loan_age` can reset later in a loan's history. To prevent later-life records from leaking into the early-life target window, construct loan age directly from the reporting month and first-payment month.


In [116]:
perf["reporting_month"] = pd.to_datetime(
    perf["monthly_reporting_period"].astype(str),
    format="%Y%m"
)

orig["first_payment_month"] = pd.to_datetime(
    orig["first_payment_date"].astype(str),
    format="%Y%m"
)


In [117]:
perf_calendar = perf.merge(
    orig[["loan_id", "first_payment_month"]],
    on="loan_id",
    how="left",
    validate="many_to_one"
)

perf_calendar["calendar_loan_age"] = (
    (perf_calendar["reporting_month"].dt.year - perf_calendar["first_payment_month"].dt.year) * 12
    + (perf_calendar["reporting_month"].dt.month - perf_calendar["first_payment_month"].dt.month)
    + 1
)


### 5.2 Validate the Reporting Timeline


In [118]:
first_records = (
    perf_calendar
    .sort_values(["loan_id", "reporting_month"])
    .groupby("loan_id", as_index=False)
    .nth(0)
)

timeline_check = first_records[
    ["loan_id", "first_payment_month", "reporting_month", "loan_age", "calendar_loan_age"]
].copy()

timeline_check.head()


,loan_id,first_payment_month,reporting_month,loan_age,calendar_loan_age
0,F19Q10000056,2019-03-01,2019-02-01,0,0
39,F19Q10000060,2019-03-01,2019-02-01,0,0
55,F19Q10000084,2019-03-01,2019-02-01,0,0
70,F19Q10000106,2019-03-01,2019-02-01,0,0
87,F19Q10000189,2019-03-01,2019-02-01,0,0


In [119]:
timeline_check[
    ["loan_age", "calendar_loan_age"]
].value_counts().sort_index()


loan_age  calendar_loan_age
0         0                    45767
1         1                     2874
2         2                      952
3         3                      209
4         4                       67
5         5                       48
6         6                       24
7         7                       23
8         8                       25
9         9                        8
10        10                       2
11        11                       1
Name: count, dtype: int64

**Observation:** Retrieving each loan's actual first chronological performance row produces internally consistent first-payment dates, reporting periods, and starting loan ages. Using `nth(0)` preserves a real row; `groupby().first()` is avoided because it can select the first non-null value independently across columns and synthesize a combination that never appeared in the data.


### 5.3 Diagnose Reported Loan-Age Resets


In [120]:
age_repeats = (
    perf_calendar.groupby(["loan_id", "loan_age"])
    .size()
    .reset_index(name="count")
)

reset_loan_ids = age_repeats.loc[
    age_repeats["count"] > 1,
    "loan_id"
].unique()

print("Repeated loan-age combinations:", (age_repeats["count"] > 1).sum())
print("Loans with evidence of reported-age resets:", len(reset_loan_ids))


Repeated loan-age combinations: 9819
Loans with evidence of reported-age resets: 889


**Observation:** A small subset of loans contains repeated reported loan ages, confirming that the supplied `loan_age` field can reset later in the performance history. Because these loans are disproportionately associated with delinquency, the final target uses calendar-derived age rather than the reported field.


### 5.4 Define the 24-Month Serious-Delinquency Target


In [121]:
perf_24m = perf_calendar[
    perf_calendar["calendar_loan_age"].between(0, 24)
].copy()

target_24m = (
    perf_24m.groupby("loan_id", as_index=False)
    .agg(max_delinquency_24m=("delinquency_numeric", "max"))
)

target_24m["seriously_delinquent_24m"] = (
    target_24m["max_delinquency_24m"] >= 3
)


In [122]:
print("Minimum calendar loan age:", perf_24m["calendar_loan_age"].min())
print("Maximum calendar loan age:", perf_24m["calendar_loan_age"].max())
print("Unique calendar loan ages:", perf_24m["calendar_loan_age"].nunique())
print("Loans in target table:", len(target_24m))


Minimum calendar loan age: 0
Maximum calendar loan age: 24
Unique calendar loan ages: 25
Loans in target table: 50000


### 5.5 Examine REO Cases


In [123]:
reo_loans = perf_calendar.loc[
    perf_calendar["delinquency_status"] == "RA",
    "loan_id"
].unique()

reo_check = (
    perf_calendar[perf_calendar["loan_id"].isin(reo_loans)]
    .groupby("loan_id", as_index=False)
    .agg(max_numeric_delinquency=("delinquency_numeric", "max"))
)

print("Loans with an RA status:", len(reo_loans))
reo_check["max_numeric_delinquency"].describe()


Loans with an RA status: 16


count    16.000000
mean     27.375000
std      10.397916
min       9.000000
25%      21.250000
50%      28.000000
75%      36.000000
max      43.000000
Name: max_numeric_delinquency, dtype: float64

**Observation:** Sixteen loans eventually entered REO (RA), and every REO loan also recorded numeric delinquency well above the 90+ threshold. Treating RA as non-numeric therefore does not hide serious-delinquency events in this sample.


### 5.6 Handle Early Data-Defect Removals


Zero-balance code 96 represents a data-defect removal. Loans ending this way before completing the 24-month calendar window have uncertain outcomes and are excluded from the target population.


In [124]:
perf_sorted = perf_calendar.sort_values(
    ["loan_id", "reporting_month"]
)

loan_followup = (
    perf_sorted.groupby("loan_id", as_index=False)
    .agg(max_calendar_age=("calendar_loan_age", "max"))
)

terminal_rows = (
    perf_sorted.groupby("loan_id", as_index=False)
    .nth(-1)[["loan_id", "zero_balance_code"]]
    .rename(columns={"zero_balance_code": "last_zero_balance_code"})
)

loan_followup = loan_followup.merge(
    terminal_rows,
    on="loan_id",
    how="left",
    validate="one_to_one"
)

early_defect_ids = set(
    loan_followup.loc[
        (loan_followup["max_calendar_age"] < 24)
        & (loan_followup["last_zero_balance_code"] == 96),
        "loan_id"
    ]
)

print("Early code-96 removals:", len(early_defect_ids))


Early code-96 removals: 103


In [125]:
target_24m_clean = target_24m[
    ~target_24m["loan_id"].isin(early_defect_ids)
].copy()

print("Original target population:", len(target_24m))
print("Clean target population:", len(target_24m_clean))
print("Excluded early defects:", len(target_24m) - len(target_24m_clean))
print(
    "Serious-delinquency rate:",
    target_24m_clean["seriously_delinquent_24m"].mean()
)


Original target population: 50000
Clean target population: 49897
Excluded early defects: 103
Serious-delinquency rate: 0.04579433633284566


**Observation:** Using calendar-based follow-up identifies 103 loans removed for code 96 before completing the 24-month window. Excluding these uncertain outcomes leaves 49,897 loans, including 2,285 serious-delinquency events, for a 24-month event rate of about 4.58%.


## 6. Create Modeling Dataset


### 6.1 Merge Origination Data and Target


In [126]:
model_data = orig.merge(
    target_24m_clean[["loan_id", "seriously_delinquent_24m"]],
    on="loan_id",
    how="inner",
    validate="one_to_one"
)

model_data.shape


(49897, 33)

### 6.2 Validate Merge


In [127]:
print("Rows:", len(model_data))
print("Unique loans:", model_data["loan_id"].nunique())
print("Duplicate loan IDs:", model_data["loan_id"].duplicated().sum())
print()
print(model_data["seriously_delinquent_24m"].value_counts())
print()
print(
    "Serious-delinquency rate:",
    model_data["seriously_delinquent_24m"].mean()
)


Rows: 49897
Unique loans: 49897
Duplicate loan IDs: 0

seriously_delinquent_24m
False    47612
True      2285
Name: count, dtype: int64

Serious-delinquency rate: 0.04579433633284566


### 6.3 Remove Unusable Features


Remove fields that are unusable in this vintage or not meaningful as predictors. VantageScore is entirely missing in the 2019 sample, and the pre-HARP loan identifier is not used as a modeling feature.


In [128]:
model_data = model_data.drop(
    columns=[
        "vantage_score",
        "pre_harp_loan_id"
    ]
)

model_data.shape


(49897, 31)

In [129]:
model_data.isna().sum().sort_values(ascending=False)


special_eligibility_program    45237
property_valuation_method      44469
msa                             4562
dti                               31
credit_score                      17
cltv                               1
harp_indicator                     0
interest_only_indicator            0
property_type                      0
first_payment_month                0
super_conforming_flag              0
seller_name                        0
num_borrowers                      0
original_loan_term                 0
loan_purpose                       0
loan_id                            0
postal_code                        0
amortization_type                  0
property_state                     0
first_payment_date                 0
prepayment_penalty                 0
channel                            0
original_interest_rate             0
ltv                                0
original_upb                       0
occupancy_status                   0
num_units                          0
m

### 6.4 Save Processed Dataset


In [130]:
model_data.to_parquet(
    "../data/processed/model_data_2019.parquet",
    index=False
)


In [131]:
check = pd.read_parquet(
    "../data/processed/model_data_2019.parquet"
)

print("Shape:", check.shape)
print("Unique loans:", check["loan_id"].nunique())
print("Serious delinquencies:", check["seriously_delinquent_24m"].sum())
print(
    "Serious-delinquency rate:",
    check["seriously_delinquent_24m"].mean()
)


Shape: (49897, 31)
Unique loans: 49897
Serious delinquencies: 2285
Serious-delinquency rate: 0.04579433633284566


**Observation:** The processed 2019 modeling dataset contains one row per mortgage with the corrected calendar-based 24-month serious-delinquency target. The saved file is the canonical 2019 dataset used by downstream analysis.
